## Sequential Chats and Customer Onboarding

## Setup

In [17]:
llm_config = {"model": "models/gemini-2.5-flash","api_key":"","api_type":"google"}

In [18]:
from autogen import ConversableAgent

## Creating the needed agents

In [19]:
from autogen import ConversableAgent

onboarding_personal_information_agent = ConversableAgent(
    name="onboarding_personal_information_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",  # Removed spaces
    system_message='''You are a helpful customer service agent.
    You are here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)


## Creating tasks
Now, you can craft a series of tasks to facilitate the onboarding process.

In [20]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

## Start the onboarding process

In [21]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)



********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product. Could you tell me your name and location?

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Devi from Chennai


customer_proxy_agent (to onboarding_personal_information_agent):

Devi from Chennai

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?
Context: 
```json
{
  "name": "Devi",
  "location": "Chennai"
}
```

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  AI Agents


customer_proxy_agent (to onboarding_topic_preference_agent):

AI Agents

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.
Context: 
```json
{
  "name": "Devi",
  "location": "Chennai"
}
```
AI Agents

--------------------------------------------------------------------------------


C:\Users\PAVAN\anaconda3\envs\AIagent\Lib\site-packages\autogen\oai\gemini.py:803: UserWarning: Cost calculation is not implemented for model models/gemini-2.5-flash. Cost will be calculated zero.
  warnings.warn(


customer_engagement_agent (to customer_proxy_agent):

Hey Devi! Fantastic to have you here! Since you're interested in AI Agents and you're from the vibrant city of Chennai, let's mix those two together for some fun!

Imagine an AI agent, let's call her 'Chellamma', specifically designed for Chennai. She wouldn't just give you the fastest route through traffic; she'd know the exact minute the *vada* stall near Marina Beach gets its fresh batch, or tell you which movie at a specific multiplex has the best AC on a hot day!

And here's a fun fact about AI agents for you: Did you know that some of the earliest concepts of what we now call AI agents can be traced back to science fiction long before computers were even a widespread thing? Writers were dreaming up helpful (and sometimes mischievous!) automatons and thinking machines that could perform tasks, learn, and even converse, much like the sophisticated AI agents being developed in places like Chennai today!

It makes you wonder, if y

## Print out the summary

In [22]:
summary=[]
for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print("\n")

{'content': '```json\n{\n  "name": "Devi",\n  "location": "Chennai"\n}\n```', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


{'content': 'AI Agents', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


{'content': 'Devi, interested in AI Agents from Chennai, was presented with the concept of a Chennai-specific AI agent ("Chellamma") capable of local, fun tasks (e.g., finding fresh *vadas*, best AC movies). The discussion also touched upon the historical roots of AI agents in science fiction, ending with a question about what uniquely "Chennai" task Devi would give such an agent.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}




In [23]:
summary[0]

{'content': '```json\n{\n  "name": "Devi",\n  "location": "Chennai"\n}\n```',
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [24]:
#!pip install pandas

In [25]:
import pandas as pd
import json

# Step 1: Extract content string
content_str = summary[0]['content']

# Step 2: Remove ```json and ```
clean_str = content_str.replace("```json", "").replace("```", "").strip()

# Step 3: Convert to dictionary
data_dict = json.loads(clean_str)

# Step 4: Convert to DataFrame
df = pd.DataFrame([data_dict])

df

,name,location
0,Devi,Chennai


In [26]:
df["TopicPreference"]=summary[1]["content"]

In [27]:
df

,name,location,TopicPreference
0,Devi,Chennai,AI Agents


In [28]:
df["Enagement"]=summary[2]["content"]

In [29]:
df

,name,location,TopicPreference,Enagement
0,Devi,Chennai,AI Agents,"Devi, interested in AI Agents from Chennai, wa..."


In [30]:
#!pip install gspread
#!pip install oauth2client

In [31]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    # Define the scope for Google Sheets API
    scope = ['https://www.googleapis.com/auth/spreadsheets']

    # Add credentials to the account
    creds = ServiceAccountCredentials.from_json_keyfile_name(
        'websiteleadchatgptapi-a46f50ae1507.json', scope
    )

    # Authorize the client
    client = gspread.authorize(creds)

    # Specify the Google Sheet ID
    sheet_id = '1NneXRit2qh2c06H-w-Hhq3H41ppvF6GjLVbW1argQvU'

    # Open the spreadsheet using its ID
    spreadsheet = client.open_by_key(sheet_id)

    # Select the first worksheet (index 0)
    worksheet = spreadsheet.get_worksheet(0)

    # Append each row of the DataFrame to the worksheet
    for _, row in df.iterrows():
        # Convert row to a list and append to the sheet
        worksheet.append_row(row.tolist())

    print("Data added successfully!")
    return "added"


In [32]:
add_leads_to_google_sheet(df)

Data added successfully!


'added'

## Print out the cost

In [33]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0, 'models/gemini-2.5-flash': {'cost': 0, 'prompt_tokens': 49, 'completion_tokens': 25, 'total_tokens': 74}}, 'usage_excluding_cached_inference': {'total_cost': 0}}


{'usage_including_cached_inference': {'total_cost': 0, 'models/gemini-2.5-flash': {'cost': 0, 'prompt_tokens': 65, 'completion_tokens': 2, 'total_tokens': 67}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'models/gemini-2.5-flash': {'cost': 0, 'prompt_tokens': 65, 'completion_tokens': 2, 'total_tokens': 67}}}


{'usage_including_cached_inference': {'total_cost': 0, 'models/gemini-2.5-flash': {'cost': 0, 'prompt_tokens': 430, 'completion_tokens': 337, 'total_tokens': 767}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'models/gemini-2.5-flash': {'cost': 0, 'prompt_tokens': 430, 'completion_tokens': 337, 'total_tokens': 767}}}


